March 2027 corn futures — six months of historical data
Adapted from futures_intro.ipynb: Databento Historical API, GLBX.MDP3,
finm37000.disk_cache, and Plotly. The outright contract is ZCH7 (H = March).
This notebook downloads individual trades, caches monthly request chunks,
and builds interactive daily price/volume and intraday charts.
Run this notebook from your FINM_37000 project using the same kernel as the template.
It needs finm37000, databento, pandas, plotly, and the cache dependencies
used by your course package (typically pyarrow). Your existing Databento key
setup is reused; no key is stored in this notebook. If third-party packages are
missing, install them in your notebook kernel with
%pip install databento pandas plotly pyarrow. The course package must already
be installed or importable from your project.
Downloads require your Databento account's historical access and may be billed.
Run the cost-estimate cell before the download cell. No market data has been
preloaded in this file.

In [1]:
import os
import sys
import pandas as pd
import databento as db
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots
from IPython.display import display
from finm37000 import disk_cache, get_databento_api_key, temp_env

pio.templates.default = "plotly_dark"
print("Kernel:", sys.executable)

# Prefer an existing environment variable, otherwise use the course helper.
api_key = os.environ.get("DATABENTO_API_KEY") or get_databento_api_key()
with temp_env(DATABENTO_API_KEY=api_key):
    client = db.Historical()
del api_key


Kernel: /home/hxu/workspace/FINM_37000/final_project/Corn_Futures_37000/finm37000_venv/bin/python


In [2]:
DATASET = "GLBX.MDP3"
SYMBOL = "ZCH7"
TIMEZONE = "America/Chicago"
CACHE_SCOPE = "zc_mar27_history"
MONTHS = 6
START_DATE = None
END_DATE = None
BAR_RULE = "1D"  # Try "1h" for hourly candles instead.
INTRADAY_RULE = "5min"

In [3]:
def chicago_timestamp(value):
    ts = pd.Timestamp(value)
    return ts.tz_localize(TIMEZONE) if ts.tzinfo is None else ts.tz_convert(TIMEZONE)

availability = client.metadata.get_dataset_range(dataset=DATASET)
trade_range = availability.get("schema", {}).get("trades", availability)
available_end = pd.to_datetime(trade_range["end"], utc=True).tz_convert(TIMEZONE)
available_start = pd.to_datetime(trade_range["start"], utc=True).tz_convert(TIMEZONE)
latest_complete_end = min(
    pd.Timestamp.now(tz=TIMEZONE).normalize(), available_end.normalize()
)
end = latest_complete_end if END_DATE is None else chicago_timestamp(END_DATE)
start = end - pd.DateOffset(months=MONTHS) if START_DATE is None else chicago_timestamp(START_DATE)
if start >= end:
    raise ValueError("START_DATE must be earlier than END_DATE.")
if start < available_start or end > available_end:
    raise ValueError("Requested range is outside Databento's available trades range. Adjust the dates.")
print(f"{SYMBOL}: [{start}, {end})")
print("Times shown in Chicago time; the end timestamp is excluded.")


ZCH7: [2026-04-12 00:00:00-05:00, 2026-10-12 00:00:00-05:00)
Times shown in Chicago time; the end timestamp is excluded.


In [4]:
@disk_cache(scope=CACHE_SCOPE)
def cached_market_data(dataset: str, symbol: str, schema: str,
                       start_iso: str, end_iso: str) -> pd.DataFrame:
    return client.timeseries.get_range(
        dataset=dataset,
        symbols=[symbol],
        stype_in="raw_symbol",
        schema=schema,
        start=start_iso,
        end=end_iso,
    ).to_df()


def monthly_windows(start, end):
    cursor = start
    while cursor < end:
        next_month = cursor.normalize().replace(day=1) + pd.DateOffset(months=1)
        stop = min(next_month, end)
        yield cursor, stop
        cursor = stop


def load_history(schema="trades"):
    chunks = []
    for left, right in monthly_windows(start, end):
        print(f"Loading {SYMBOL} {schema}: {left.isoformat()} -> {right.isoformat()}")
        chunk = cached_market_data(
            DATASET, SYMBOL, schema, left.isoformat(), right.isoformat()
        )
        if not chunk.empty:
            chunks.append(chunk)
    if not chunks:
        return pd.DataFrame()
    # Half-open windows do not overlap. Preserve separate trades that share a timestamp.
    return pd.concat(chunks).sort_index(kind="stable")


In [5]:
# Optional metadata preview. This does not download trade records.
# The estimate covers the full range; cached chunks will not be downloaded again.
estimated_cost = client.metadata.get_cost(
    dataset=DATASET, symbols=[SYMBOL], stype_in="raw_symbol",
    schema="trades", start=start.isoformat(), end=end.isoformat(),
)
print(f"Estimated cost if the entire range is uncached: ${estimated_cost:.4f}")


Estimated cost if the entire range is uncached: $0.0000


In [ ]:
trades = load_history("trades")
if trades.empty:
    raise ValueError("No trades returned. Check the symbol, dates, and account access.")
if "symbol" in trades and set(trades["symbol"].dropna()) != {SYMBOL}:
    raise ValueError("The response contains an unexpected contract symbol.")
if not {"price", "size"}.issubset(trades.columns):
    raise ValueError("Trade data must include price and size columns.")
print(f"Loaded {len(trades):,} trade records")
print(f"DataFrame memory: {trades.memory_usage(deep=True).sum() / 1024**2:,.1f} MiB")
display(trades.head())


Loading ZCH7 trades: 2026-04-12T00:00:00-05:00 -> 2026-05-01T00:00:00-05:00
Loading ZCH7 trades: 2026-05-01T00:00:00-05:00 -> 2026-06-01T00:00:00-05:00


/tmp/ipykernel_20486/3106072862.py:4: BentoWarning: The streaming request contained one or more days which have reduced quality: 2026-05-24 (degraded). See: https://databento.com/docs/api-reference-historical/metadata/metadata-get-dataset-condition
  return client.timeseries.get_range(


Loading ZCH7 trades: 2026-06-01T00:00:00-05:00 -> 2026-07-01T00:00:00-05:00
Loading ZCH7 trades: 2026-07-01T00:00:00-05:00 -> 2026-08-01T00:00:00-05:00


In [ ]:
def trades_by_event_time(df):
    out = df.copy()
    source = out["ts_event"] if "ts_event" in out.columns else out.index
    out.index = pd.DatetimeIndex(pd.to_datetime(source, utc=True)).tz_convert(TIMEZONE)
    out.index.name = "event_time"
    return out.sort_index(kind="stable")


def make_trade_bars(df, rule="1D"):
    if df.empty:
        return pd.DataFrame(columns=["open", "high", "low", "close", "volume", "trade_count"])
    # Invalid/nonpositive trade records do not contribute to the chart.
    valid = df.loc[df["price"].notna() & df["size"].notna()
                   & df["price"].gt(0) & df["size"].gt(0)]
    prices = valid["price"].resample(rule).ohlc()
    prices["volume"] = valid["size"].resample(rule).sum()
    prices["trade_count"] = valid["size"].resample(rule).count()
    return prices.dropna(subset=["open", "high", "low", "close"])


def price_volume_chart(bars, title):
    if bars.empty:
        raise ValueError("No valid trades in this chart range.")
    fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
                        vertical_spacing=0.05, row_heights=[0.72, 0.28])
    fig.add_trace(go.Candlestick(
        x=bars.index, open=bars["open"], high=bars["high"],
        low=bars["low"], close=bars["close"], name="Trade price",
    ), row=1, col=1)
    fig.add_trace(go.Bar(x=bars.index, y=bars["volume"], name="Volume"), row=2, col=1)
    fig.update_layout(title=title, height=750, showlegend=False,
                      xaxis_rangeslider_visible=False, hovermode="x unified")
    fig.update_yaxes(title_text="Price (cents/bushel)", row=1, col=1)
    fig.update_yaxes(title_text="Contracts", row=2, col=1)
    fig.update_xaxes(title_text="Chicago time", row=2, col=1)
    return fig

# Keep the received records in `trades`; build chart data separately.
trades_ct = trades_by_event_time(trades)
bars = make_trade_bars(trades_ct, BAR_RULE)
print(f"{len(bars):,} bars; {bars['volume'].sum():,.0f} contracts traded")
display(bars.tail())
fig = price_volume_chart(bars, f"{SYMBOL} — March 2027 corn: {BAR_RULE} price and volume")
fig.show()


In [ ]:
INTRADAY_DATE = None  # Example: "2026-10-09"
selected_day = (trades_ct.index.max().normalize() if INTRADAY_DATE is None
                else chicago_timestamp(INTRADAY_DATE).normalize())
next_day = selected_day + pd.DateOffset(days=1)
one_day = trades_ct.loc[(trades_ct.index >= selected_day) & (trades_ct.index < next_day)]
intraday_bars = make_trade_bars(one_day, INTRADAY_RULE)
intraday_fig = price_volume_chart(
    intraday_bars, f"{SYMBOL} — {selected_day.date()}, {INTRADAY_RULE} bars"
)
intraday_fig.show()
